In [1]:

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from xgboost import XGBClassifier

# 1. Gerar dados sintéticos e reproduzíveis
rng = np.random.default_rng(42)
n = 5000

latencia = np.clip(
    rng.gamma(2.2, 22, n), 1, 300
)
perda = np.clip(
    rng.beta(1.2, 10, n) * 100, 0, 100
)
jitter = np.clip(
    rng.gamma(1.8, 7, n), 0.1, 100
)
disponibilidade = np.clip(
    100 - perda * 0.7 + rng.normal(0, 1.8, n),
    70, 100
)

X = pd.DataFrame({
    "latencia_ms": latencia,
    "perda_pct": perda,
    "jitter_ms": jitter,
    "disponibilidade_pct": disponibilidade
})

# 2. Criar classes didáticas por regras
risco = (
    (latencia > 100).astype(int)
    + (perda > 3).astype(int)
    + (jitter > 25).astype(int)
    + (disponibilidade < 96).astype(int)
)

y = np.where(
    risco >= 3, "Com Falha",
    np.where(risco >= 1, "Em Risco", "Estável")
)

# Introduzir 3% de ruído nos rótulos
mascara = rng.random(n) < 0.03
classes = np.array(["Estável", "Em Risco", "Com Falha"])

for i in np.where(mascara)[0]:
    y[i] = rng.choice(classes[classes != y[i]])

# 3. Mesma divisão para todos os modelos
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20,
    random_state=42, stratify=y
)

encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)

modelos = {
    "Árvore de Decisão": DecisionTreeClassifier(
        max_depth=8, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=12,
        random_state=42, n_jobs=1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=5,
        learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42, n_jobs=1,
        tree_method="hist"
    )
}

resultados = []

for nome, modelo in modelos.items():
    # Treinamento
    inicio = time.perf_counter()

    if nome == "XGBoost":
        modelo.fit(X_treino, y_treino_num)
    else:
        modelo.fit(X_treino, y_treino)

    tempo_treino = time.perf_counter() - inicio

    # Previsão e medição repetida
    modelo.predict(X_teste.iloc[:10])
    repeticoes = 100

    inicio = time.perf_counter()
    for _ in range(repeticoes):
        previsoes = modelo.predict(X_teste)

    tempo_previsao = (
        time.perf_counter() - inicio
    ) / (repeticoes * len(X_teste)) * 1000

    if nome == "XGBoost":
        previsoes = encoder.inverse_transform(
            previsoes.astype(int)
        )

    resultados.append({
        "Modelo": nome,
        "Acurácia (%)": round(
            accuracy_score(y_teste, previsoes) * 100, 2
        ),
        "Precisão macro (%)": round(
            precision_score(
                y_teste, previsoes,
                average="macro", zero_division=0
            ) * 100, 2
        ),
        "Recall macro (%)": round(
            recall_score(
                y_teste, previsoes,
                average="macro", zero_division=0
            ) * 100, 2
        ),
        "F1 macro (%)": round(
            f1_score(
                y_teste, previsoes,
                average="macro", zero_division=0
            ) * 100, 2
        ),
        "Treinamento (s)": round(tempo_treino, 4),
        "Previsão (ms/amostra)": round(tempo_previsao, 6)
    })

print(pd.DataFrame(resultados).to_string(index=False))


           Modelo  Acurácia (%)  Precisão macro (%)  Recall macro (%)  F1 macro (%)  Treinamento (s)  Previsão (ms/amostra)
Árvore de Decisão          95.9               94.90             92.32         93.56           0.0779               0.002575
    Random Forest          97.0               97.20             93.64         95.33           2.2734               0.043416
          XGBoost          96.2               95.95             92.46         94.12           0.6321               0.035906
